In [1]:
import torch
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
from datasets import load_dataset

ds = load_dataset("EE21/ToS-Summaries")

In [3]:
split_ds=ds['train'].train_test_split(test_size=0.2,seed=42)

In [4]:
split_ds

DatasetDict({
    train: Dataset({
        features: ['plain_text', 'summary'],
        num_rows: 720
    })
    test: Dataset({
        features: ['plain_text', 'summary'],
        num_rows: 181
    })
})

In [5]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

MODEL_NAME="facebook/bart-base"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

In [ ]:
import re

def remove_html_tags(text):
    pattern=re.compile('<.*?>')
    return pattern.sub(r'',text)

def preprocess_text(text):
    text=text.strip()
    text=remove_html_tags(text)
    return text

In [7]:
PREFIX="Summarize the following policy.\n\n"

def process_dataset(example):
    prompt=PREFIX+example['plain_text']
    
    prompt=preprocess_text(prompt)

    model_inputs=tokenizer(
        prompt,
        max_length=1024,
        truncation=True
    )
    
    labels=tokenizer(
        text_target=example['summary'],
        max_length=256,
        truncation=True,
    )
    
    model_inputs['labels']=labels['input_ids']
    return model_inputs

In [8]:
tokenized_ds=split_ds.map(process_dataset,remove_columns=ds['train'].column_names)

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [10]:
# from peft import LoraConfig, get_peft_model, TaskType

# lora_config=LoraConfig(
#     task_type=TaskType.SEQ_2_SEQ_LM,
#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.1,
#     target_modules="all-linear",
#     bias="none"
# )

In [11]:
# model=get_peft_model(model,lora_config)
# model.print_trainable_parameters()

In [12]:
import evaluate
import numpy as np

rouge=evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds,labels=eval_pred
    
    decode_preds=tokenizer.batch_decode(preds,skip_special_tokens=True)
    labels=np.where(labels!=-100,labels,tokenizer.pad_token_id)
    decode_labels=tokenizer.batch_decode(labels,skip_special_tokens=True)
    
    result=rouge.compute(
        predictions=decode_preds,
        references=decode_labels,
        use_stemmer=True
    )
    
    return {
    "rouge1": result["rouge1"],
    "rouge2": result["rouge2"],
    "rougeL": result["rougeL"],
}

### Model Training

In [13]:
from torch.utils.data import DataLoader

train_dataload=DataLoader(
    tokenized_ds['train'],
    batch_size=8,
    pin_memory=True,
    shuffle=True,
    collate_fn=data_collator
)

test_dataload=DataLoader(
    tokenized_ds['test'],
    batch_size=8,
    pin_memory=True,
    shuffle=False,
    collate_fn=data_collator
)

In [14]:
from tqdm.auto import tqdm
import torch

class BartFineTuned:
    def __init__(self):
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(self.device)
        self.tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
        self.best_loss=float("inf")
        
    def parameters(self):
        return self.model.parameters()
    
    def compile(self,optimizer):
        self.optimizer=optimizer
        
        
    def fit(self,dataload,epochs,validation_dataloader):
        for epoch in range(1,epochs+1):
            self.model.train()
            print(f"Epoch {epoch} {'-'*100}")
            train_loss=0
            progress_bar=tqdm(range(len(dataload)))
            
            for batch in dataload:
                
                batch=batch.to(self.device)
                outputs=self.model(**batch)
                loss=outputs.loss
                
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                progress_bar.update(1)
                train_loss+=loss.item()
            
            val_loss=self.evaluate(validation_dataloader)
            
            if val_loss<self.best_loss:
                self.best_loss=val_loss
                self.model.save_pretrained("models/bart_fine_tuned")
                self.tokenizer.save_pretrained("models/bart_fine_tuned")
            
            print("Train Loss : ",train_loss/len(dataload),"\nValidation Loss : ",val_loss)
    
    def evaluate(self,dataloader):
        self.model.eval()

        with torch.no_grad():
            val_loss=0
            for batch in dataloader:
                batch=batch.to(self.device)
                output=self.model(**batch)
                loss=output.loss
                val_loss+=loss.item()
                
            return val_loss/len(dataloader)

In [16]:
from torch.optim import AdamW
trainer=BartFineTuned()
trainer.compile(AdamW(trainer.parameters(),lr=5e-5))
trainer.fit(train_dataload,epochs=3,validation_dataloader=test_dataload)

Epoch 1 ----------------------------------------------------------------------------------------------------


  0%|          | 0/90 [00:00<?, ?it/s]

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\transformers\modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Train Loss :  1.7471100277370877 
Validation Loss :  0.8450904903204545
Epoch 2 ----------------------------------------------------------------------------------------------------


  0%|          | 0/90 [00:00<?, ?it/s]

Train Loss :  0.82972079316775 
Validation Loss :  0.5938304766364719
Epoch 3 ----------------------------------------------------------------------------------------------------


  0%|          | 0/90 [00:00<?, ?it/s]

Train Loss :  0.6207284020053015 
Validation Loss :  0.5306858057561128
